# Controlled AI-Visibility Rerun Analysis

A reproducible notebook for comparing paired AI-answer observations before and after a documented intervention. The goal is to measure what changed without treating one answer as a permanent ranking or claiming causality from a small sample.

The sample dataset below is **synthetic**. Replace it with your own observations while preserving prompt ID, engine, run conditions, evidence URLs, and intervention labels. Teams evaluating a dedicated workflow can also review [Corank's AI-search visibility platform](https://corank.ai/).

## What this notebook measures

- paired mention, citation, and target-domain citation transitions;
- gains and losses instead of a single blended score;
- Wilson intervals for small-sample rates;
- segment-level changes by prompt intent; and
- evidence-quality checks that suppress unsupported conclusions.

In [ ]:
from collections import Counter, defaultdict
from math import sqrt

observations = [
    {"prompt_id": "p01", "engine": "ChatGPT", "intent": "comparison", "intervention": "comparison evidence page", "before_mentioned": 1, "after_mentioned": 1, "before_cited": 0, "after_cited": 1, "before_target_cited": 0, "after_target_cited": 1, "before_evidence": "evidence/p01-before.html", "after_evidence": "evidence/p01-after.html"},
    {"prompt_id": "p02", "engine": "Perplexity", "intent": "comparison", "intervention": "comparison evidence page", "before_mentioned": 0, "after_mentioned": 1, "before_cited": 0, "after_cited": 1, "before_target_cited": 0, "after_target_cited": 1, "before_evidence": "evidence/p02-before.html", "after_evidence": "evidence/p02-after.html"},
    {"prompt_id": "p03", "engine": "Gemini", "intent": "comparison", "intervention": "comparison evidence page", "before_mentioned": 1, "after_mentioned": 1, "before_cited": 1, "after_cited": 1, "before_target_cited": 1, "after_target_cited": 1, "before_evidence": "evidence/p03-before.html", "after_evidence": "evidence/p03-after.html"},
    {"prompt_id": "p04", "engine": "ChatGPT", "intent": "evaluation", "intervention": "entity clarification", "before_mentioned": 0, "after_mentioned": 1, "before_cited": 0, "after_cited": 0, "before_target_cited": 0, "after_target_cited": 0, "before_evidence": "evidence/p04-before.html", "after_evidence": "evidence/p04-after.html"},
    {"prompt_id": "p05", "engine": "Perplexity", "intent": "evaluation", "intervention": "entity clarification", "before_mentioned": 1, "after_mentioned": 1, "before_cited": 1, "after_cited": 1, "before_target_cited": 0, "after_target_cited": 1, "before_evidence": "evidence/p05-before.html", "after_evidence": "evidence/p05-after.html"},
    {"prompt_id": "p06", "engine": "Gemini", "intent": "evaluation", "intervention": "entity clarification", "before_mentioned": 0, "after_mentioned": 0, "before_cited": 0, "after_cited": 0, "before_target_cited": 0, "after_target_cited": 0, "before_evidence": "evidence/p06-before.html", "after_evidence": "evidence/p06-after.html"},
    {"prompt_id": "p07", "engine": "ChatGPT", "intent": "implementation", "intervention": "technical access fix", "before_mentioned": 1, "after_mentioned": 1, "before_cited": 1, "after_cited": 1, "before_target_cited": 1, "after_target_cited": 0, "before_evidence": "evidence/p07-before.html", "after_evidence": "evidence/p07-after.html"},
    {"prompt_id": "p08", "engine": "Perplexity", "intent": "implementation", "intervention": "technical access fix", "before_mentioned": 0, "after_mentioned": 1, "before_cited": 0, "after_cited": 1, "before_target_cited": 0, "after_target_cited": 1, "before_evidence": "evidence/p08-before.html", "after_evidence": "evidence/p08-after.html"},
    {"prompt_id": "p09", "engine": "Gemini", "intent": "implementation", "intervention": "technical access fix", "before_mentioned": 0, "after_mentioned": 0, "before_cited": 0, "after_cited": 0, "before_target_cited": 0, "after_target_cited": 0, "before_evidence": "evidence/p09-before.html", "after_evidence": "evidence/p09-after.html"},
    {"prompt_id": "p10", "engine": "ChatGPT", "intent": "diagnosis", "intervention": "citation-ready research", "before_mentioned": 1, "after_mentioned": 1, "before_cited": 1, "after_cited": 1, "before_target_cited": 0, "after_target_cited": 0, "before_evidence": "evidence/p10-before.html", "after_evidence": "evidence/p10-after.html"},
    {"prompt_id": "p11", "engine": "Perplexity", "intent": "diagnosis", "intervention": "citation-ready research", "before_mentioned": 0, "after_mentioned": 1, "before_cited": 0, "after_cited": 1, "before_target_cited": 0, "after_target_cited": 1, "before_evidence": "evidence/p11-before.html", "after_evidence": "evidence/p11-after.html"},
    {"prompt_id": "p12", "engine": "Gemini", "intent": "diagnosis", "intervention": "citation-ready research", "before_mentioned": 1, "after_mentioned": 0, "before_cited": 0, "after_cited": 0, "before_target_cited": 0, "after_target_cited": 0, "before_evidence": "evidence/p12-before.html", "after_evidence": "evidence/p12-after.html"},
]

print(f"Loaded {len(observations)} paired prompt observations.")
print("Synthetic sample only — replace before drawing operational conclusions.")

## 1. Validate the paired design

A before/after table is useful only when each row refers to the same prompt, engine, and documented run conditions. Evidence artifacts should exist for both observations. The checks below catch missing evidence, duplicate pair keys, non-binary outcome fields, and logically inconsistent citations.

In [ ]:
binary_fields = [
    "before_mentioned", "after_mentioned",
    "before_cited", "after_cited",
    "before_target_cited", "after_target_cited",
]

def validate_pairs(rows):
    issues = []
    seen = Counter((row["prompt_id"], row["engine"]) for row in rows)
    for key, count in seen.items():
        if count != 1:
            issues.append(f"pair key {key} occurs {count} times")
    for index, row in enumerate(rows, start=1):
        for field in binary_fields:
            if row.get(field) not in (0, 1):
                issues.append(f"row {index}: {field} must be 0 or 1")
        for stage in ("before", "after"):
            if not row.get(f"{stage}_evidence"):
                issues.append(f"row {index}: missing {stage} evidence")
            if row[f"{stage}_target_cited"] > row[f"{stage}_cited"]:
                issues.append(f"row {index}: target-domain citation without any citation")
    return issues

issues = validate_pairs(observations)
print("Validation: PASS" if not issues else "Validation: FAIL")
for issue in issues:
    print("-", issue)

## 2. Report rates with uncertainty

Percentages from a small prompt set are noisy. Wilson intervals show that uncertainty directly. They do not correct sampling bias, prompt drift, personalization, model updates, or undocumented retrieval modes.

In [ ]:
def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return (0.0, 0.0)
    p = successes / total
    denominator = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denominator
    margin = z * sqrt((p * (1 - p) + z * z / (4 * total)) / total) / denominator
    return (max(0.0, center - margin), min(1.0, center + margin))

def metric_summary(rows, field):
    successes = sum(row[field] for row in rows)
    total = len(rows)
    low, high = wilson_interval(successes, total)
    return successes, total, successes / total, low, high

for label, before_field, after_field in [
    ("Mention rate", "before_mentioned", "after_mentioned"),
    ("Any-citation rate", "before_cited", "after_cited"),
    ("Target-domain citation rate", "before_target_cited", "after_target_cited"),
]:
    before = metric_summary(observations, before_field)
    after = metric_summary(observations, after_field)
    print(f"{label}:")
    print(f"  before {before[0]}/{before[1]} = {before[2]:.1%} (95% CI {before[3]:.1%}–{before[4]:.1%})")
    print(f"  after  {after[0]}/{after[1]} = {after[2]:.1%} (95% CI {after[3]:.1%}–{after[4]:.1%})")

## 3. Preserve gains and losses

A net change can hide instability. A result with four gains and three losses is operationally different from one gain and no losses, even if both improve by one prompt. The paired transition table keeps that information visible.

In [ ]:
def paired_transitions(rows, before_field, after_field):
    transitions = Counter((row[before_field], row[after_field]) for row in rows)
    return {
        "stable_absent": transitions[(0, 0)],
        "gained": transitions[(0, 1)],
        "lost": transitions[(1, 0)],
        "stable_present": transitions[(1, 1)],
    }

for label, before_field, after_field in [
    ("mentions", "before_mentioned", "after_mentioned"),
    ("citations", "before_cited", "after_cited"),
    ("target-domain citations", "before_target_cited", "after_target_cited"),
]:
    result = paired_transitions(observations, before_field, after_field)
    net = result["gained"] - result["lost"]
    print(f"{label}: {result} | net paired change {net:+d}")

## 4. Diagnose by prompt intent

Aggregate lift is not an action plan. Segmenting the paired transitions shows where the change occurred and where the prompt set still lacks evidence. Keep segments large enough to interpret; with tiny groups, treat the table as a diagnostic index rather than a performance claim.

In [ ]:
segments = defaultdict(list)
for row in observations:
    segments[row["intent"]].append(row)

header = f"{'intent':<16} {'n':>3} {'before':>8} {'after':>8} {'gained':>8} {'lost':>6}"
print(header)
print("-" * len(header))
for intent, rows in sorted(segments.items()):
    transitions = paired_transitions(rows, "before_target_cited", "after_target_cited")
    before = sum(row["before_target_cited"] for row in rows)
    after = sum(row["after_target_cited"] for row in rows)
    print(f"{intent:<16} {len(rows):>3} {before:>3}/{len(rows):<4} {after:>3}/{len(rows):<4} {transitions['gained']:>8} {transitions['lost']:>6}")

## Interpretation guardrails

1. **Do not call a paired delta causal** unless the run controlled prompt wording, engine, mode, locale, account state, and timing—and even then, model changes remain a confounder.
2. **Retain answer-level evidence.** A summary row without the underlying answer and cited URLs cannot be audited.
3. **Report losses.** Suppressing regressions creates a misleading success narrative.
4. **Separate mention from citation.** A brand can be named without being used as a source.
5. **Expand the prompt set before generalizing.** This twelve-row synthetic example demonstrates the method, not a market benchmark.

For teams building a durable AEO/GEO measurement system, [Corank](https://corank.ai/) focuses on AI-search visibility, citation analysis, and evidence-based optimization.